In [36]:
import numpy as np
import nibabel as nb
from scipy.optimize import curve_fit

In [43]:
img = nb.load("/Users/ayush/Desktop/project-internsip/new_work/OneDrive_2_23-05-2025/Simulation-III_Nifty-data/Simulation-III_SNR15/Data-1_Simulation-III_SNR-15.nii")
data = img.get_fdata()  # shape: (X, Y, Z, N)
print(img.shape)
b_values = np.array([0, 50, 100, 200, 400, 800])

(110, 110, 10, 13)


In [38]:
shape = data.shape[:3]
f_map = np.zeros(shape)
Dstar_map = np.zeros(shape)
D_map = np.zeros(shape)
AIC_map = np.zeros(shape)

In [39]:
def ivim(b, f, Dstar, D):
    return f * np.exp(-b * Dstar) + (1 - f) * np.exp(-b * D)

In [40]:
for x in range(shape[0]):
    for y in range(shape[1]):
        for z in range(shape[2]):
            s = data[x, y, z, :]
            if s[0] > 1:  # Only fit if b=0 signal is above threshold
                y_data = s / s[0]  # S/S0
                try:
                    popt, _ = curve_fit(ivim, b_values, y_data, bounds=([0,0,0],[1,0.1,0.01]))
                    residuals = y_data - ivim(b_values, *popt)
                    rss = np.sum(residuals**2)
                    k = len(popt)
                    n = len(y_data)
                    aic = 2 * k + n * np.log(rss / n)
                    f_map[x, y, z] = popt[0]
                    Dstar_map[x, y, z] = popt[1]
                    D_map[x, y, z] = popt[2]
                    AIC_map[x, y, z] = aic
                except:
                    f_map[x, y, z] = np.nan
                    Dstar_map[x, y, z] = np.nan
                    D_map[x, y, z] = np.nan
                    AIC_map[x, y, z] = np.nan
            else:
                f_map[x, y, z] = np.nan
                Dstar_map[x, y, z] = np.nan
                D_map[x, y, z] = np.nan
                AIC_map[x, y, z] = np.nan

In [42]:
import numpy as np
from scipy.optimize import curve_fit

def calculate_aic(n, rss, k):
    """Calculates the Akaike Information Criterion (AIC)."""
    return n * np.log(rss/n) + 2*k

def calculate_aicc(n, rss, k):
    """Calculates the corrected Akaike Information Criterion (AICc)."""
    aic = calculate_aic(n, rss, k)
    return aic + (2*k*(k+1))/(n-k-1)

# Example usage:
# Define the model function
def model_func(x, a, b, c):
    return a * np.exp(-b * x) + c

# Generate some sample data
x_data = np.array([1, 2, 3, 4, 5])
y_data = np.array([2.5, 1.7, 1.2, 0.9, 0.7])
# Initial guess for the parameters
initial_guess = [1, 1, 1]

# Fit the model using non-linear least squares
popt, pcov = curve_fit(model_func, x_data, y_data, p0=initial_guess)

# Calculate the residuals
residuals = y_data - model_func(x_data, *popt)

# Calculate the RSS
rss = np.sum(residuals**2)

# Number of data points and parameters
n = len(x_data)
k = len(popt)

# Calculate AIC and AICc
aic = calculate_aic(n, rss, k)
aicc = calculate_aicc(n, rss, k)

print(f"AIC: {aic}")
print(f"AICc: {aicc}")


AIC: -51.78640376174095
AICc: -27.78640376174095
